# L6.3 · Flow matching: caminar con la brújula

| Notebook | Qué abre |
|---|---|
| L6.1 | El problema: por qué la regresión cae en el medio, y por qué tampoco sirve aprender "ruido → acción" de un salto |
| L6.2 | El camino recto y la brújula: qué velocidad aprende la red y cómo se entrena |
| **L6.3** | **La inferencia: caminar desde el ruido de a pasitos** |
| después: L7 | Un VLA (SmolVLA) desarmado pieza por pieza: su experto de acciones es esta misma brújula |

En L6.2 la red aprendió a decir "para dónde moverse" en cada punto y momento,
pero todavía no generó nada. Acá la usamos. Esto es lo que hacen π0 y SmolVLA
cada vez que tienen que mover el robot: sus "10 pasadas" son los pasitos de
este notebook.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from toolkit.flow import matching, toy

plt.rcParams["figure.dpi"] = 110
torch.set_num_threads(4)  # redes chiquitas: más núcleos sólo agregan coordinación
torch.manual_seed(0)

demos = toy.sample_demos(400)
# La brújula de L6.2, entrenada igual: el código quedó en toolkit/flow/matching.py.
compass = matching.train_velocity(demos)


def wide_view(ax):
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.6, 3.0)


def tree_fraction(points):
    distance = torch.linalg.vector_norm(points - torch.tensor(toy.TREE_CENTER), dim=1)
    return (distance < toy.TREE_RADIUS).float().mean().item()

## 1. Caminar: seguir las flechas de a pasitos

Arrancamos en ruido, en `t = 0`, y repetimos:

1. le preguntamos a la brújula la velocidad en el punto y momento actuales;
2. nos movemos un pasito en esa dirección: `x ← x + dt · v`;
3. avanzamos el reloj: `t ← t + dt`.

Con `N` pasos, `dt = 1 / N`, y después de `N` vueltas llegamos a `t = 1`. A
este método se lo llama **Euler**: el más simple que hay, y el que usan
SmolVLA y π0.

La diferencia con L6.2: **esta vez nadie le dice a cada punto adónde ir.** No
hay pareja. Cada punto sólo tiene la brújula.

In [ ]:
def generate(compass, noise, steps, keep_path=False):
    """Camina desde el ruido hasta t = 1 siguiendo la brújula, en `steps` pasos de Euler."""
    x = noise.clone()
    path = [x.clone()]
    dt = 1.0 / steps
    with torch.no_grad():
        for i in range(steps):
            t = torch.full((len(x), 1), i * dt)
            x = x + dt * compass(x, t)  # el pasito
            path.append(x.clone())
    return torch.stack(path) if keep_path else x


noise = torch.randn(300, 2)
path = generate(compass, noise, steps=10, keep_path=True)  # (11 posiciones, 300 puntos, 2)
colors = toy.side_colors(path[-1])

fig, ax = plt.subplots(figsize=(6.5, 6))
toy.draw_scene(ax, title="300 puntos caminando 10 pasos con la brújula", alpha=0.1)
wide_view(ax)
for i in range(path.shape[1]):
    ax.plot(path[:, i, 0], path[:, i, 1], color=colors[i], lw=0.6, alpha=0.5)
ax.scatter(path[0, :, 0], path[0, :, 1], s=6, color="gray", zorder=3, label="ruido (t = 0)")
ax.scatter(path[-1, :, 0], path[-1, :, 1], s=8, c=colors, zorder=4, label="donde llegaron (t = 1)")
ax.legend(loc="lower left", fontsize=8)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(19, 3.6))
for ax, step in zip(axes, [0, 2, 4, 6, 8, 10]):
    toy.draw_scene(ax, title=f"después de {step} pasos (t = {step / 10})")
    wide_view(ax)
    ax.scatter(path[step, :, 0], path[step, :, 1], s=5, c=colors)
plt.tight_layout()

Tres cosas para mirar:

- **Los caminos se curvan.** Las parejas de L6.2 iban en línea recta; los
  caminos generados no. Suben hacia el árbol (al principio la brújula apunta
  al promedio) y a mitad de camino se abren hacia un lado.
- **El ruido decide el lado desde el principio.** En el primer cuadro de la
  tira, los colores (que son el destino final) ya están separados por una
  línea casi vertical: el ruido que cae a la izquierda del eje termina a la
  izquierda. La brújula no tiene nada de azar; todo el azar está en el punto
  de partida, como en la escalera de L6.1.
- **Nadie le dijo a ningún punto adónde ir**, y llegan a los dos grupos.

## 2. ¿Llegaron a lugares válidos?

Generamos 4000 puntos con 10 pasos y contamos:

In [ ]:
samples = generate(compass, torch.randn(4000, 2), steps=10)
left = (samples[:, 0] < 0).float().mean().item()
print(f"izquierda {left:.1%} · derecha {1 - left:.1%} · en el árbol {tree_fraction(samples):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
toy.draw_scene(axes[0], title="4000 puntos generados (10 pasos)")
axes[0].scatter(samples[:, 0], samples[:, 1], s=2, c=toy.side_colors(samples), alpha=0.4)
bins = np.linspace(-1.8, 1.8, 73)
axes[1].hist(demos[:, 0], bins=bins, density=True, color="gray", alpha=0.6, label="demostraciones")
axes[1].hist(samples[:, 0], bins=bins, density=True, histtype="step", color="red", lw=1.5, label="generados")
axes[1].axvspan(-toy.TREE_RADIUS, toy.TREE_RADIUS, color="forestgreen", alpha=0.15)
axes[1].set_xlabel("posición lateral")
axes[1].legend(fontsize=8)
plt.tight_layout()

Mitad y mitad, como las demostraciones, y la forma de cada grupo copia la de
los datos. Un poco más del 1 % cae en el árbol: son los puntos que arrancaron
justo sobre la línea divisoria, que la brújula no termina de mandar a ningún
lado. Con más pasos baja (sección 3).

### La prueba que L1 no pasaba: 70 % por la izquierda

En L6.1, con 70 % de demostraciones por la izquierda, L1 elegía la izquierda
**siempre**. Entrenamos una brújula con esos datos y generamos:

In [ ]:
demos_70 = toy.sample_demos(400, left_fraction=0.7, seed=1)
compass_70 = matching.train_velocity(demos_70)
samples_70 = generate(compass_70, torch.randn(4000, 2), steps=10)
left_70 = (samples_70[:, 0] < 0).float().mean().item()
print(f"izquierda {left_70:.1%} · derecha {1 - left_70:.1%} · en el árbol {tree_fraction(samples_70):.2%}")

fig, ax = plt.subplots(figsize=(6, 4.5))
toy.draw_scene(ax, demos_70, title="70 % por la izquierda: lo que genera la brújula", alpha=0.15)
ax.scatter(samples_70[:1000, 0], samples_70[:1000, 1], s=4, color="red", alpha=0.5, label="1000 generados")
ax.legend(loc="lower left", fontsize=8)

Cerca de 70/30: respeta las proporciones (la diferencia de un par de puntos
es del entrenamiento y del sorteo). Donde L1 elegía siempre la mayoría, flow
matching sortea: la minoría aparece con su frecuencia.

## 3. ¿Cuántos pasos hacen falta?

Mismo ruido, distinta cantidad de pasos:

In [ ]:
shared_noise = torch.randn(2000, 2)
step_counts = [1, 2, 3, 5, 10, 50]
fig, axes = plt.subplots(1, 6, figsize=(19, 3.6))
in_tree = {}
for ax, steps in zip(axes, step_counts):
    result = generate(compass, shared_noise, steps)
    in_tree[steps] = tree_fraction(result)
    toy.draw_scene(ax, title=f"{steps} paso{'s' if steps > 1 else ''} · {in_tree[steps]:.0%} en el árbol")
    ax.scatter(result[:, 0], result[:, 1], s=2, c=toy.side_colors(result), alpha=0.4)
plt.tight_layout()
print({steps: f"{fraction:.1%}" for steps, fraction in in_tree.items()})

Con 1 paso, **todo** cae en el tronco. Con 2, los puntos se desparraman en
una línea entre los dos grupos: el primer paso los llevó a mitad de camino del
promedio y el segundo no alcanzó para abrirlos. Con 5 ya hay dos grupos; con
10 queda un 2 % en el árbol, y con 50 casi nada. Cada paso extra cuesta una
pasada más de la red, así que hay que elegir: SmolVLA y π0 usan 10.

### Por qué un solo paso es la regresión

Al principio (`t = 0`) la brújula ideal apunta al promedio de las
demostraciones, y con un largo justo: `promedio - x` (L6.2, sección 4). Un solo
paso de tamaño 1 hace `x + (promedio - x) = promedio`, para **cualquier** ruido.
Flow matching con un paso es exactamente la regresión de L6.1. Los pasos son
los que le dan la oportunidad de decidir en el camino.

## 4. Cuánto cuesta: cada paso es una pasada de la red

Cada paso le pregunta a la red una vez. Diez pasos, diez pasadas:

In [ ]:
batch = torch.randn(4000, 2)
for steps in [1, 10, 50]:
    start = time.perf_counter()
    generate(compass, batch, steps)
    print(f"{steps:>3} pasos: {1000 * (time.perf_counter() - start):6.1f} ms")

En el juguete es nada. En un VLA, la red es un transformer de cientos de
millones de parámetros, y el robot espera la acción. Por eso SmolVLA (lo
verificamos en su código) calcula **una sola vez** la parte cara, la que mira
la imagen, la instrucción y el estado, y guarda el resultado. En cada una de
las 10 pasadas sólo trabaja el experto de acciones, que es más chico y lee lo
que quedó guardado. Lo vamos a ver por dentro en L7.

## 5. Para profundizar: la brújula perfecta copia, la red generaliza

¿Qué pasa si caminamos con la brújula **ideal** de L6.2 en vez de con la red?
Comparamos las dos con 100 pasos, haciendo zoom en el grupo de la izquierda:

In [ ]:
def ideal_compass(x, t):
    return matching.ideal_velocity(x, t[0].item(), demos)


zoom_noise = torch.randn(1000, 2)
from_ideal = generate(ideal_compass, zoom_noise, steps=100)
from_net = generate(compass, zoom_noise, steps=100)


def distance_to_nearest_demo(points):
    return torch.cdist(points, demos).min(dim=1).values.median().item()


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, result, name in zip(axes, [from_ideal, from_net], ["brújula ideal", "red entrenada"]):
    ax.scatter(demos[:, 0], demos[:, 1], s=40, facecolors="none", edgecolors="black", lw=0.6, label="demostraciones")
    ax.scatter(result[:, 0], result[:, 1], s=4, color="red", alpha=0.6, label="generados")
    ax.set_xlim(-1.5, -0.5)
    ax.set_ylim(1.5, 2.5)
    ax.set_aspect("equal")
    ax.set_title(f"{name} · distancia típica a la demostración más cercana {distance_to_nearest_demo(result):.3f}")
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()

La brújula ideal **copia**: cada punto generado cae prácticamente encima de
una demostración. No inventa nada; es una máquina de sortear demostraciones de
la lista. La red, por ser imperfecta y suave, rellena el grupo con puntos
nuevos, parecidos pero no iguales a los de las demostraciones. Esa
imperfección es lo que llamamos generalizar. Y es una advertencia: una red que
se ajusta "demasiado bien" a pocas demostraciones termina repitiéndolas de
memoria.

## Lo que nos llevamos

- Generar es caminar: desde ruido, `N` pasitos siguiendo la brújula. Euler,
  la cuenta más simple: `x ← x + dt · v`.
- Los caminos generados se curvan: arrancan hacia el promedio y se abren hacia
  un lado. El ruido inicial ya decide el lado; la brújula no tiene azar.
- Llegan a lugares válidos y **respetan las proporciones** (70/30), que era lo
  que ni MSE ni L1 podían hacer.
- Un solo paso es la regresión de L6.1. Con 10 pasos ya anda: por eso SmolVLA
  y π0 usan 10.
- Cada paso es una pasada de la red: los VLA calculan una vez la parte cara y
  repiten sólo el experto de acciones.

### Del juguete al VLA

| En el juguete | En SmolVLA |
|---|---|
| un punto de 2 números | un bloque de 50 acciones, cada una de 32 números (las articulaciones, rellenadas hasta 32) |
| la brújula: una red chica que mira (punto, momento) | el experto de acciones: un transformer que además mira la imagen, la instrucción y el estado |
| 10 pasitos de Euler | sus 10 pasadas (`num_steps = 10`) |
| `t = 0` ruido → `t = 1` acción | al revés: `t = 1` ruido → `t = 0` acción; la misma cuenta dada vuelta |

## Referencias

- Lipman et al., [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747)
  (2023).
- Lipman et al., [Flow Matching Guide and Code](https://arxiv.org/abs/2412.06264)
  (2024): Euler y métodos mejores para caminar.
- Black et al., [π0](https://arxiv.org/abs/2410.24164) (2024): las 10 pasadas
  y el experto de acciones.
- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025): el VLA
  que vamos a desarmar en L7.